# Lucia v1.1 9B: the non-commercial line

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/qwen-symbolic-thinking/blob/main/notebooks/lucia_v11_9b_nc.ipynb)

The fine-tune of [Qwen/Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B) on the rows of Lucia v1.1 4B's non-commercial
version, held near the base model's behavior while it learns them: the same training set and the same recipe as that release,
on the larger base, with the general rows Qwen3.5-9B answered itself. This notebook runs the training (the general-row KL at
weight 1), then its screen, probe, museum and preservation reads.

The training is near a day long, longer than a runtime lives. It runs with resume on (package v0.1.63): the whole training state
saves to the private twin every 2 hours and once more after the runtime's 23rd hour. If the runtime ends mid-run, open this
notebook on a fresh runtime and run it again: the lane sees the run's marker still at "started" and continues the training from
its saved state, in the same folder, on the same path.

Runtime: a GPU (the RTX PRO 6000) and the `HF_TOKEN` secret with read access to the accepted rows and write access to the private
twin `AbstractPhil/qwen-symbolic-thinking-image-training-private`, where every file of these runs lives. Run C1, restart the session (Runtime > Restart session), then run C2. Each run
leaves `runs/done/<name>.json` in the twin; the queue cell ends after 45 idle minutes.

In [ ]:
# C1: install, then Runtime > Restart session
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.63", "transformers>=5.0", "accelerate>=1.0", "peft>=0.18")

FAST_PATH = True   # the linear-attention kernels Qwen3.5 uses; causal-conv1d compiles for about 5 minutes; all or nothing
if FAST_PATH:
    if all(importlib.util.find_spec(m) for m in ("causal_conv1d", "fla", "triton")):
        print("fast path already present")
    else:
        try:
            pip("causal-conv1d", "--no-build-isolation")
            pip("flash-linear-attention", "triton")
            print("fast path on")
        except subprocess.CalledProcessError as e:
            print("fast path off (training still runs, slower):", e)
            subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "causal-conv1d"])

try:                # Colab's torchao 0.10 breaks PEFT when it builds a LoRA (PEFT wants 0.16 or newer, or none at all)
    import torchao
    _v = tuple(int(x) for x in torchao.__version__.split(".")[:2])
    if _v < (0, 16):
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
        print("torchao", torchao.__version__, "removed: PEFT cannot build a LoRA beside it")
    else:
        print("torchao", torchao.__version__, "stays")
except ImportError:
    print("no torchao")

print("installed; now Runtime > Restart session, then run C2")

In [ ]:
# C2: work through this line's queue (runs/queue_lucia_v11_9b_nc.json in the private twin; new runs can be added while it runs)
import os
os.environ["QST_REPO"] = "AbstractPhil/qwen-symbolic-thinking-image-training-private"   # before the package is imported: every file of these runs stays private
LANE = "lucia_v11_9b_nc"

from qwen_symbolic_thinking import REPO, __version__
from qwen_symbolic_thinking.queue import run_queue

print("package", __version__, "| project repo", REPO, "| lane", LANE)
ran = run_queue(LANE, wait_hours=12)
print("ran:", ran)
